# Semantic Chunking：按“语义断点”切分文档，而不是按固定长度硬切

本节目标：实现语义切分（Semantic Chunking）的最小版本，理解为什么它常常比固定长度切分更适合检索。

## 核心直觉

固定长度切分（比如每 400 字符一块）会在任意位置截断句子/段落，导致：

- 命中 chunk 可能只包含半句或半段
- 语义被稀释，检索/生成都更难

**语义切分（Semantic Chunking）** 的思路是：

- 先把文本拆成更细粒度的单元（通常是句子）
- 用 embeddings 计算相邻单元之间“语义变化”的大小
- 在“变化足够大”的位置作为断点切开，让每个 chunk 更像自然段/主题块

本节使用 LangChain 的 `SemanticChunker`（位于 `langchain-experimental`）。

## 0) 导入依赖并加载环境变量

延续前面 notebook 的约定：从 `../.env` 读取 `OPENAI_API_KEY`。

依赖说明：

- 语义切分用 `SemanticChunker`（来自 `langchain-experimental`）
- embeddings 用 `OpenAIEmbeddings`
- 向量库用 `FAISS`
- PDF 抽取用 `pypdf`

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv("../.env")

# 让 `all_rag_techniques/` 子目录里的 notebook
# 也能导入上一级目录的 `helper_functions.py`
sys.path.insert(0, str(Path("..").resolve()))

from pypdf import PdfReader

from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.vectorstores import FAISS

from helper_functions import retrieve_context_per_question, show_context

/tmp/ipykernel_3984335/1622463165.py:19: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


## 1) 下载 PDF 并抽取为纯文本

我们使用示例文件 `Understanding_Climate_Change.pdf`。把每页文本提取出来并拼接成一个大字符串 `content`。

PDF 抽取不是重点；我们只需要得到可用的正文文本。

In [2]:
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7890"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"
os.environ["ALL_PROXY"] = "http://127.0.0.1:7890"

In [3]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

PDF_URL = "https://raw.githubusercontent.com/NirDiamant/RAG_Techniques/main/data/Understanding_Climate_Change.pdf"
PDF_PATH = DATA_DIR / "Understanding_Climate_Change.pdf"

if not PDF_PATH.exists():
    resp = requests.get(
        PDF_URL,
        timeout=60,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    resp.raise_for_status()
    PDF_PATH.write_bytes(resp.content)

reader = PdfReader(str(PDF_PATH))
pages_text = [(p.extract_text() or "") for p in reader.pages]
content = "\n\n".join(pages_text)

assert len(content.strip()) > 0, "PDF 提取到的文本为空（pypdf.extract_text 可能失败）"
print("Pages:", len(reader.pages))
print("Chars:", len(content))

Pages: 33
Chars: 72592


## 2) 语义切分的断点策略

`SemanticChunker` 支持三种断点策略：

- **`percentile`**：计算相邻句子差异，超过第 X 分位数就切
- **`standard_deviation`**：超过 X 个标准差就切
- **`interquartile`**：用四分位距决定切分点

本节使用：

- `breakpoint_threshold_type='percentile'`
- `breakpoint_threshold_amount=90`

直觉：这会让切分更“保守”——只有语义变化非常明显的位置才切开，从而形成更像自然段/主题块的 chunks。

In [4]:
text_splitter = SemanticChunker(
    OpenAIEmbeddings(model="text-embedding-3-small"),
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,
)

docs = text_splitter.create_documents([content])
print("Num semantic chunks:", len(docs))
print("\n--- Semantic chunk 0 preview ---\n")
print(docs[0].page_content[:800])

Num semantic chunks: 63

--- Semantic chunk 0 preview ---

Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the planet's overall weather patterns, including temperature, 
precipitation, and wind patterns, over an extended period. Over the past century, human 
activities, particularly the burning of fossil fuels and deforestation, have significantly 
contributed to climate change. Historical Context 
The Earth's climate has changed throughout history. Over the past 650,000 years, there have 
been seven cycles of glacial advance and retreat, with the abrupt end of the last ice age about 
11,700 years ago marking the beginning of the modern climate era and human civilization. Most of these climate changes are a


## 3) 建立向量库与 retriever

我们把语义 chunks 做成可检索的向量索引：

- 用 embeddings 给语义 chunks 建索引
- 用 FAISS 做向量库
- `k=2` 返回最相关的两块语义 chunk

检索采用 retriever 的新式调用：`retriever.invoke(query)`（`helper_functions.retrieve_context_per_question` 内部也是这么做的）。

In [5]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, embeddings)
chunks_query_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

## 4) 测试检索效果

我们用一个简单问题来检查“语义切分 + 检索”的结果是否更连贯：

- `"What is the main cause of climate change?"`

直接取回 top-2 的语义 chunks 并打印出来。

In [6]:
test_query = "What is the main cause of climate change?"
context = retrieve_context_per_question(test_query, chunks_query_retriever)
show_context(context)

Context 1:
Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the planet's overall weather patterns, including temperature, 
precipitation, and wind patterns, over an extended period. Over the past century, human 
activities, particularly the burning of fossil fuels and deforestation, have significantly 
contributed to climate change. Historical Context 
The Earth's climate has changed throughout history. Over the past 650,000 years, there have 
been seven cycles of glacial advance and retreat, with the abrupt end of the last ice age about 
11,700 years ago marking the beginning of the modern climate era and human civilization. Most of these climate changes are attributed to very small variations in Earth's orbit that 
change the amount of solar energy our planet receives. During the Holocene epoch, which 
began at the end of the last ice age, huma

## 5) 小结

- **语义切分** 的目标是让每个 chunk 更像自然段/主题块，而不是固定长度硬切
- 它通常能减少“命中半句/半段”的情况，从源头提升检索可用性
- 和 window-around-chunk 的关系：
  - 语义切分是 **索引阶段** 优化（先把 chunk 切得更合理）
  - window-around-chunk 是 **检索后处理**（命中后再补齐邻居上下文）